# Hommelgeluid classificeren: buzz, flight of ruis

**Datalab III – Lectoraat Smart Sensor Systems (opdrachtgever: dr. Sam Aerts)**

Centrale onderzoeksvraag: *Hoe nauwkeurig herkent een classificatiemodel het gedrag van hommels in geluidsfragmenten van één seconde uit de proefkas van Tomatoworld?*

Dit notebook doorloopt de deliverables uit de opdracht met de data die we nu hebben (het featurebestand en het labelbestand, zonder wav-bestanden):

| Stap | Deliverable uit de opdracht | Waar in dit notebook |
|---|---|---|
| 1 | Data-inventarisatie | Hoofdstuk 1 |
| 2 | Preprocessing (opschonen, klassen, vensters) | Hoofdstuk 2 |
| 3 | Features | Hoofdstuk 3 |
| 4 | Uitlegbare baseline | Hoofdstuk 5 |
| 5 | Machine learning-modellen | Hoofdstuk 6 |
| 6 | Evaluatie (accuracy, precision, recall, F1, confusion matrix) | Hoofdstuk 7 |
| 7 | Visualisatie met voorspelde labels | Hoofdstuk 8 |
| 8 | Conclusie, foutanalyse en advies | Hoofdstuk 9 |

**Normen uit het plan van aanpak** (op een testset die niet voor training is gebruikt):
- Stap 1 (hommel of ruis): recall van hommelgeluid ≥ 0,90
- Stap 2 (buzz of flight): accuracy ≥ 90% en precision van buzz ≥ 0,90

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.inspection import permutation_importance
from sklearn.metrics import (f1_score, accuracy_score, precision_score, recall_score,
                             classification_report, confusion_matrix, ConfusionMatrixDisplay)

SEED = 42
KLASSEN = ["buzz", "flight", "ruis"]
FEATURES_PAD = "../data/processed_data/bee_features_per_second.csv"
LABELS_PAD = "../data/raw_labels/buzzdetection_labels.csv"

sns.set_theme(style="whitegrid")
pd.set_option("display.precision", 3)

## 1. Data-inventarisatie

We hebben twee bestanden:
- `bee_features_per_second.csv`: per seconde audio 64 features (MFCC's, zero-crossing rate, energie, spectral centroid en rolloff), gemaakt met `preprocessdata.py`.
- `buzzdetection_labels.csv`: de handmatige labels van de masterstudent, één rij per seconde.

In [ ]:
features = pd.read_csv(FEATURES_PAD)
labels = pd.read_csv(LABELS_PAD)

print("Featurebestand:", features.shape)
print("Labelbestand:  ", labels.shape)
print("Aantal wav-bestanden:", features["source_file"].nunique())
print("Ontbrekende waarden in de features:", features.drop(columns="label").isna().sum().sum())

**Duplicaten.** `preprocessdata.py` zoekt naar `*.WAV` én `*.wav`. Op Windows/macOS (hoofdletterongevoelig) wordt elk bestand daardoor twee keer ingelezen. We controleren dit:

In [ ]:
dubbel = features.duplicated(subset=["source_file", "start"]).sum()
print(f"Dubbele rijen (zelfde bestand + zelfde seconde): {dubbel}")

df = features.drop_duplicates(subset=["source_file", "start"]).copy()
df = df.sort_values(["source_file", "start"]).reset_index(drop=True)
print(f"Unieke seconden na opschonen: {len(df)}")
print(f"Waarvan gelabeld: {df['label'].notna().sum()}  |  ongelabeld: {df['label'].isna().sum()}")

**Koppeling labels ↔ audio.** Niet elk label hoort bij een seconde waarvoor audio bestaat:

In [ ]:
# labeltijd '2026-03-24 11:13:01:00' -> laatste ':00' (subseconde) weghalen
labels["start_sec"] = labels["start"].str.rsplit(":", n=1).str[0]
gekoppeld = labels["start_sec"].isin(df["start"])
print(f"Labels met bijbehorende audio:   {gekoppeld.sum()}")
print(f"Labels zonder audio (vervallen): {(~gekoppeld).sum()}")

In [ ]:
df["dag"] = pd.to_datetime(df["start"]).dt.date
overzicht = df.groupby("dag").agg(
    wav_bestanden=("source_file", "nunique"),
    seconden=("start", "size"),
    gelabeld=("label", "count"),
)
overzicht

In [ ]:
gelabeld = df[df["label"].notna()]
fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
gelabeld["label"].value_counts().plot.bar(ax=ax[0], color="steelblue")
ax[0].set_title("Originele labels (seconden)")
ax[0].set_ylabel("aantal seconden")
(gelabeld.groupby("source_file")["label"].apply(lambda s: s.isin(["buzz", "flight"]).sum())
    .sort_values(ascending=False).head(15).plot.bar(ax=ax[1], color="darkorange"))
ax[1].set_title("Seconden met hommelgeluid per wav-bestand (top 15)")
plt.tight_layout(); plt.show()

**Wat valt op?**
- Hommelgeluid is zeldzaam: er zijn veel meer seconden met radio of stemmen dan met buzz of flight. De dataset is dus sterk **onbalans**; accuracy alleen zegt daardoor weinig.
- Het hommelgeluid zit in een **klein aantal wav-bestanden**. Een paar bestanden bevatten het grootste deel van alle buzz-seconden. Dat maakt het lastig om goed te generaliseren.

## 2. Preprocessing

Keuzes (met onderbouwing):

1. **Duplicaten verwijderd** (zie hierboven).
2. **Klassen samengevoegd tot drie klassen**, zoals in de opdracht en het plan van aanpak: `buzz`, `flight` en `ruis` (= radio + voice + noise). Voor het lectoraat is alleen van belang óf er een hommel is en wat die doet; het type ruis maakt niet uit.
3. **Tijdvenster = 1 seconde.** De features en labels zijn al per seconde. Kortere vensters (100–500 ms) kunnen pas als we de wav-bestanden opnieuw verwerken.
4. **Normalisatie** gebeurt in de modellen zelf met `StandardScaler` (alleen op de trainingsdata gefit, dus geen datalek).

In [ ]:
df["klasse"] = df["label"].map({"buzz": "buzz", "flight": "flight"})
df.loc[df["label"].notna() & df["klasse"].isna(), "klasse"] = "ruis"
df["klasse"].value_counts()

## 3. Features

**Bestaande features (64):** 20 MFCC-gemiddelden, 20 MFCC-standaardafwijkingen, 20 delta's, zero-crossing rate, rms (energie), spectral centroid en spectral rolloff.

**Keuze:** we gebruiken alle 64 features. De delta's lijken weinig te zeggen (hun gemiddelde ligt bij alle klassen rond 0), maar weglaten maakte het beste model (SVM) in de kruisvalidatie iets slechter. Ook hebben we context-features getest (gemiddelde over de vorige, huidige en volgende seconde). Dat gaf geen stabiele verbetering op ongeziene bestanden, dus houden we het eenvoudig.

> FFT-features zoals dominante frequentie en bandenergie (500–1000 Hz, 1000–1500 Hz, …) uit de opdracht kunnen alleen uit de wav-bestanden worden berekend. Dat staat als volgende stap in het advies.

In [ ]:
BASIS = [c for c in features.columns if c not in ["start", "end", "label", "source_file"]]

FEATURES = BASIS
print(f"Aantal features: {len(FEATURES)}")

**Verschillen tussen de klassen (deelvraag 2).** De spectral centroid is het feature dat het dichtst bij ‘de toonhoogte’ komt:

In [ ]:
lab = df[df["klasse"].notna()].reset_index(drop=True)

fig, ax = plt.subplots(1, 3, figsize=(14, 3.8))
for a, kol, titel in zip(ax, ["spectral_centroid_mean", "spectral_rolloff_mean", "rms_mean"],
                        ["Spectral centroid (Hz)", "Spectral rolloff (Hz)", "Energie (rms)"]):
    sns.boxplot(data=lab, x="klasse", y=kol, order=KLASSEN, ax=a, showfliers=False)
    a.set_title(titel); a.set_xlabel(""); a.set_ylabel("")
plt.tight_layout(); plt.show()

lab.groupby("klasse")[["spectral_centroid_mean", "spectral_rolloff_mean", "rms_mean", "zcr_mean"]].median()

Buzz heeft gemiddeld een **hogere** spectral centroid dan flight. Dat sluit aan bij de literatuur (Morgan et al., 2016) en spreekt de README tegen (die noemt buzz ‘laagfrequent’). De verdelingen overlappen wel sterk, dus één feature alleen zal niet genoeg zijn.

## 4. Train/test-split per wav-bestand

Seconden uit hetzelfde bestand lijken sterk op elkaar. Als die zowel in train als test zitten, lijkt het model beter dan het is (**datalek**). Daarom splitsen we **per wav-bestand** met `StratifiedGroupKFold`: ongeveer 20% van de bestanden gaat naar de testset, die we pas aan het eind gebruiken.

In [ ]:
X = lab[FEATURES]
y = lab["klasse"].values
groepen = lab["source_file"].values

splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
train_idx, test_idx = next(splitter.split(X, y, groepen))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y[train_idx], y[test_idx]
g_train, g_test = groepen[train_idx], groepen[test_idx]

assert set(g_train).isdisjoint(g_test)  # geen bestand in beide sets
print(f"Train: {len(y_train)} seconden uit {len(set(g_train))} bestanden")
print(f"Test:  {len(y_test)} seconden uit {len(set(g_test))} bestanden")
pd.DataFrame({"train": pd.Series(y_train).value_counts(), "test": pd.Series(y_test).value_counts()})

**Waarom dit belangrijk is:** hetzelfde model met een willekeurige split (zonder groepen) versus een split per bestand:

In [ ]:
demo = make_pipeline(StandardScaler(), SVC(class_weight="balanced"))
p_willekeurig = cross_val_predict(demo, X_train, y_train, cv=StratifiedKFold(5, shuffle=True, random_state=SEED))
p_groep = cross_val_predict(demo, X_train, y_train, groups=g_train,
                            cv=StratifiedGroupKFold(5, shuffle=True, random_state=SEED))
print(f"Macro-F1 willekeurige split: {f1_score(y_train, p_willekeurig, average='macro'):.3f}  (te optimistisch)")
print(f"Macro-F1 split per bestand:  {f1_score(y_train, p_groep, average='macro'):.3f}  (eerlijk)")

## 5. Baseline: drempelwaarden op de spectral centroid

De baseline is bewust simpel en volledig uitlegbaar: een beslisboom van maximaal 2 niveaus op **alleen** de spectral centroid. Dat zijn gewoon twee drempelwaarden in Hz. Elk ML-model moet hier duidelijk boven zitten.

In [ ]:
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)

baseline = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=SEED)
p_base = cross_val_predict(baseline, X_train[["spectral_centroid_mean"]], y_train, groups=g_train, cv=cv)
print(f"Baseline macro-F1 (kruisvalidatie): {f1_score(y_train, p_base, average='macro'):.3f}\n")

baseline.fit(X_train[["spectral_centroid_mean"]], y_train)
print("Regels van de baseline:")
print(export_text(baseline, feature_names=["spectral_centroid"]))

## 6. Machine learning-modellen vergelijken

We vergelijken vijf gangbare modellen met **5-voudige kruisvalidatie per bestand** op de trainingsset. Hoofdmetric is **macro-F1**: het gemiddelde van de F1 per klasse, zodat de kleine klassen buzz en flight even zwaar meetellen als ruis.

Alle modellen krijgen `class_weight="balanced"` (waar mogelijk), zodat fouten op de zeldzame klassen zwaarder wegen.

In [ ]:
modellen = {
    "Logistische regressie": make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000, class_weight="balanced")),
    "Random Forest": RandomForestClassifier(n_estimators=300, class_weight="balanced_subsample", random_state=SEED, n_jobs=-1),
    "SVM": make_pipeline(StandardScaler(), SVC(class_weight="balanced")),
    "k-NN": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)),
    "Gradient Boosting": HistGradientBoostingClassifier(class_weight="balanced", random_state=SEED),
}

def scores(y_echt, y_pred):
    f1 = f1_score(y_echt, y_pred, average=None, labels=KLASSEN)
    return {"macro-F1": f1_score(y_echt, y_pred, average="macro"),
            "F1 buzz": f1[0], "F1 flight": f1[1], "F1 ruis": f1[2]}

resultaten = {"Baseline (centroid)": scores(y_train, p_base)}
for naam, model in modellen.items():
    pred = cross_val_predict(model, X_train, y_train, groups=g_train, cv=cv)
    resultaten[naam] = scores(y_train, pred)

vergelijking = pd.DataFrame(resultaten).T.sort_values("macro-F1", ascending=False)
vergelijking

**Twee modelopzetten (deelvraag 3).** Uit het kennismakingsgesprek kwam het idee van een *tweestapsmodel*: eerst ‘hommel of ruis’, daarna ‘buzz of flight’. We vergelijken dat met het drie-klassenmodel, met hetzelfde algoritme (SVM).

In [ ]:
class TweeStapsModel(BaseEstimator, ClassifierMixin):
    """Stap 1: hommel vs ruis. Stap 2: buzz vs flight (alleen voor 'hommel')."""
    def __init__(self, stap1, stap2):
        self.stap1 = stap1
        self.stap2 = stap2

    def fit(self, X, y):
        y = np.asarray(y)
        self.stap1_ = clone(self.stap1).fit(X, np.where(y == "ruis", "ruis", "hommel"))
        hommel = y != "ruis"
        self.stap2_ = clone(self.stap2).fit(X[hommel], y[hommel])
        self.classes_ = np.array(KLASSEN)
        return self

    def predict(self, X):
        pred = self.stap1_.predict(X).astype(object)
        is_hommel = pred == "hommel"
        if is_hommel.any():
            pred[is_hommel] = self.stap2_.predict(X[is_hommel])
        return pred

svm = make_pipeline(StandardScaler(), SVC(class_weight="balanced"))
p_twee = cross_val_predict(TweeStapsModel(svm, svm), X_train, y_train, groups=g_train, cv=cv)
p_drie = cross_val_predict(svm, X_train, y_train, groups=g_train, cv=cv)

pd.DataFrame({"Drie klassen (SVM)": scores(y_train, p_drie),
              "Twee stappen (SVM)": scores(y_train, p_twee)}).T

**Keuze: SVM met standaardinstellingen, drie klassen.** De SVM haalt de hoogste macro-F1 en het drie-klassenmodel werkt beter dan de tweestapsopzet. Hyperparameters tunen hebben we geprobeerd, maar met zo weinig bestanden met hommelgeluid vallen de verschillen tussen instellingen binnen de spreiding tussen folds (zie hoofdstuk 7). Een getuned model scoorde op nieuwe bestanden zelfs iets slechter. We houden daarom de standaardinstellingen (`C=1`, `gamma="scale"`).

## 7. Eindevaluatie op de testset

Het gekozen model wordt nu één keer getraind op de volledige trainingsset en getest op de bestanden die het nooit heeft gezien. Ter vergelijking testen we de baseline op dezelfde testset.

In [ ]:
eindmodel = clone(modellen["SVM"]).fit(X_train, y_train)
y_pred = eindmodel.predict(X_test)
y_pred_base = baseline.predict(X_test[["spectral_centroid_mean"]])

print("=== Eindmodel (SVM) ===")
print(classification_report(y_test, y_pred, labels=KLASSEN, digits=3))
print("=== Baseline ===")
print(classification_report(y_test, y_pred_base, labels=KLASSEN, digits=3, zero_division=0))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
for a, pred, titel in [(ax[0], y_pred_base, "Baseline"), (ax[1], y_pred, "Eindmodel (SVM)")]:
    ConfusionMatrixDisplay.from_predictions(y_test, pred, labels=KLASSEN, ax=a, cmap="Blues", colorbar=False)
    a.set_title(titel); a.set_xlabel("voorspeld"); a.set_ylabel("werkelijk"); a.grid(False)
plt.tight_layout(); plt.show()

**Foutanalyse.** Welke soorten geluid worden met elkaar verward? We kijken naar het oorspronkelijke label (vóór het samenvoegen) tegenover de voorspelling:

In [ ]:
fouten = pd.crosstab(lab.iloc[test_idx]["label"].values, y_pred,
                     rownames=["origineel label"], colnames=["voorspeld"])
fouten

**Toetsing aan de normen uit het plan van aanpak:**

In [ ]:
echt_hommel = y_test != "ruis"
pred_hommel = y_pred != "ruis"

# stap 2: alleen seconden die echt hommel zijn én door het model als hommel herkend
beide = echt_hommel & pred_hommel

normen = pd.DataFrame([
    ["Stap 1: recall hommel", recall_score(echt_hommel, pred_hommel), 0.90],
    ["Stap 2: accuracy buzz/flight", accuracy_score(y_test[beide], y_pred[beide]), 0.90],
    ["Stap 2: precision buzz", precision_score(y_test, y_pred, labels=["buzz"], average="micro", zero_division=0), 0.90],
    ["Macro-F1 (alle klassen)", f1_score(y_test, y_pred, average="macro"), None],
], columns=["Metric", "Waarde", "Norm"])
normen["Gehaald?"] = normen.apply(lambda r: "" if pd.isna(r["Norm"]) else ("ja" if r["Waarde"] >= r["Norm"] else "nee"), axis=1)
print(f"(stap 2 berekend op {beide.sum()} seconden die stap 1 goed als hommel herkende)")
normen

**Robuustheid: hoe betrouwbaar is één testset?** De testset bevat maar 19 seconden buzz en 23 seconden flight. Eén paar bestanden meer of minder verandert de score flink. Daarom herhalen we de evaluatie vijf keer: elke keer is een ander deel van de bestanden de testset, zodat elk bestand precies één keer getest wordt. Zo zien we de spreiding.

In [ ]:
rij = []
for fold, (tr, te) in enumerate(splitter.split(X, y, groepen), start=1):
    m = clone(modellen["SVM"]).fit(X.iloc[tr], y[tr])
    b = clone(baseline).fit(X.iloc[tr][["spectral_centroid_mean"]], y[tr])
    p, pb = m.predict(X.iloc[te]), b.predict(X.iloc[te][["spectral_centroid_mean"]])
    rij.append({"fold": fold,
                "buzz/flight in test": int((y[te] != "ruis").sum()),
                "baseline macro-F1": f1_score(y[te], pb, average="macro"),
                "SVM macro-F1": f1_score(y[te], p, average="macro"),
                "SVM recall hommel": recall_score(y[te] != "ruis", p != "ruis"),
                "SVM precision buzz": precision_score(y[te], p, labels=["buzz"], average="micro", zero_division=0)})
robuust = pd.DataFrame(rij).set_index("fold")
robuust.loc["gemiddelde"] = robuust.mean()
robuust.loc["std"] = robuust.iloc[:5].std()
robuust

**Welke features gebruikt het model het meest?** Permutation importance: hoeveel daalt de macro-F1 op de testset als we één feature door elkaar husselen?

In [ ]:
imp = permutation_importance(eindmodel, X_test, y_test, scoring="f1_macro",
                             n_repeats=5, random_state=SEED, n_jobs=-1)
top = pd.Series(imp.importances_mean, index=FEATURES).sort_values(ascending=False).head(12)
top[::-1].plot.barh(figsize=(7, 4.5), color="seagreen")
plt.title("Top 12 features (daling macro-F1)"); plt.tight_layout(); plt.show()

## 8. Visualisatie: audio met voorspelde labels

Echte spectrogrammen vragen de wav-bestanden. Wat we wél hebben zijn de 20 MFCC's per seconde; die vormen samen een **grove spectrale weergave** (laag = lage frequenties, hoog = hoge). Hieronder per testbestand met hommelgeluid: de MFCC's over de tijd (gestandaardiseerd; rood = hoger dan gemiddeld, blauw = lager), met daaronder het echte en het voorspelde label per seconde.

In [ ]:
KLEUR = {"buzz": "#d62728", "flight": "#1f77b4", "ruis": "#d9d9d9"}
MFCC = [f"mfcc_{i}_mean" for i in range(1, 21)]

test_df = lab.iloc[test_idx].copy()
test_df["voorspeld"] = y_pred

def toon_bestand(bestand):
    d = test_df[test_df["source_file"] == bestand]
    fig, ax = plt.subplots(3, 1, figsize=(12, 4.2), sharex=True,
                           gridspec_kw={"height_ratios": [4, 0.6, 0.6]})
    z = (d[MFCC] - lab[MFCC].mean()) / lab[MFCC].std()   # per coëfficiënt standaardiseren
    ax[0].imshow(z.T.values, aspect="auto", origin="lower", cmap="coolwarm", vmin=-2.5, vmax=2.5)
    ax[0].set_ylabel("MFCC 1–20"); ax[0].set_title(bestand)
    for a, kol in [(ax[1], "klasse"), (ax[2], "voorspeld")]:
        a.imshow([[list(KLASSEN).index(k) for k in d[kol]]], aspect="auto",
                 cmap=plt.matplotlib.colors.ListedColormap([KLEUR[k] for k in KLASSEN]), vmin=0, vmax=2)
        a.set_yticks([0]); a.set_yticklabels(["echt" if kol == "klasse" else "voorspeld"])
    for a in ax: a.grid(False)
    ax[2].set_xlabel("gelabelde seconden in het bestand (op volgorde)")
    handles = [plt.Rectangle((0, 0), 1, 1, color=KLEUR[k]) for k in KLASSEN]
    fig.legend(handles, KLASSEN, loc="upper right", ncol=3)
    plt.tight_layout(); plt.show()

met_hommel = test_df.groupby("source_file")["klasse"].apply(lambda s: (s != "ruis").sum())
for bestand in met_hommel[met_hommel > 0].sort_values(ascending=False).index[:4]:
    toon_bestand(bestand)

**Toepassing op ongelabelde data.** Het model wordt nu getraind op álle gelabelde seconden en voorspelt de seconden die nog geen label hebben. Zo ziet het lectoraat waar mogelijk hommelgeluid zit; die fragmenten zijn goede kandidaten om als eerste te laten labelen.

In [ ]:
definitief = clone(modellen["SVM"]).fit(X, y)
ongelabeld = df[df["klasse"].isna()].copy()
ongelabeld["voorspeld"] = definitief.predict(ongelabeld[FEATURES])

print(f"Ongelabelde seconden: {len(ongelabeld)}")
display(ongelabeld["voorspeld"].value_counts().to_frame("aantal seconden"))
kandidaten = (ongelabeld[ongelabeld["voorspeld"] != "ruis"]
              .groupby("source_file").size().sort_values(ascending=False).head(10))
print("Bestanden met de meeste voorspelde hommelseconden (eerst laten labelen):")
kandidaten.to_frame("voorspelde hommelseconden")

## 9. Conclusie en advies

### Antwoord op de onderzoeksvraag
Met de huidige features en labels herkent het model hommelgedrag **matig**. Over vijf verschillende testsets haalt de SVM een macro-F1 van gemiddeld **0,59** (spreiding 0,46 tot 0,71). Dat is ruim twee keer zo goed als de baseline op alleen de spectral centroid (0,29), maar **de normen uit het plan van aanpak worden niet gehaald**:

| Norm | Gemiddeld over 5 testsets | Vaste testset | Norm |
|---|---|---|---|
| Stap 1: recall hommel | 0,73 | 0,62 | ≥ 0,90 |
| Stap 2: accuracy buzz/flight | – | 0,81 (26 seconden) | ≥ 0,90 |
| Stap 2: precision buzz | 0,39 | 0,26 | ≥ 0,90 |

### Belangrijkste keuzes
- **Drie klassen** (buzz, flight, ruis): werkte beter dan het tweestapsmodel (macro-F1 0,59 tegen 0,52 in de kruisvalidatie).
- **SVM met standaardinstellingen**: beste van de vijf geteste modellen. Random Forest scoorde verrassend slecht omdat het de kleine klassen bijna nooit voorspelt.
- **Split per wav-bestand**: een willekeurige split geeft een te rooskleurig beeld (0,63 tegen 0,59).

### Foutanalyse: waar gaat het mis?
1. **Radio wordt aangezien voor hommel.** In de testset werden 62 radio-seconden als buzz of flight voorspeld. Stemmen worden juist bijna altijd goed herkend als ruis. Radiogeluid lijkt in de MFCC's dus op hommelgeluid.
2. **Buzz wordt gemist.** Ongeveer de helft van de buzz-seconden wordt als ruis gezien. Waarschijnlijk duurt een buzz korter dan een seconde en ‘verdwijnt’ hij in het gemiddelde over die seconde.
3. **Te weinig hommeldata.** Er zijn maar 250 seconden buzz en flight, verdeeld over een paar bestanden. De score hangt daardoor sterk af van welke bestanden in de testset zitten.

### Advies aan het lectoraat (vervolgstappen)
1. **FFT-features berekenen uit de wav-bestanden**: dominante frequentie, bandenergie in 500–1000, 1000–1500, 1500–2000 en 2000–2500 Hz en piekamplitude. Volgens de literatuur zit het verschil tussen buzz en flight vooral in de grondfrequentie, en die zit niet in het huidige featurebestand.
2. **Kortere tijdvensters** (250 of 500 ms) zodat korte buzzes niet worden uitgemiddeld.
3. **Gericht extra labelen**: begin met de bestanden die het model nu als ‘veel hommel’ aanwijst (zie hoofdstuk 8). Dat levert sneller extra buzz- en flight-seconden op dan willekeurig labelen.
4. **Labels controleren** met een tweede beoordelaar (Cohen's kappa), zoals gepland in het plan van aanpak.
5. **Per recorder vergelijken** (16 kHz en 384 kHz) zodra bekend is welk bestand van welke recorder komt (deelvraag 4).

### Beperkingen
- Geen echte spectrogrammen: de wav-bestanden zaten niet in deze analyse; de MFCC-weergave is een grove benadering.
- De testset is klein (19 seconden buzz). Daarom rapporteren we ook het gemiddelde over vijf testsets.
- De labels zijn door één persoon gezet en (nog) niet gecontroleerd.